# 记忆压缩

上一章，智能体有了记忆，但也留下一个麻烦：聊得越久，每次要发送的聊天记录就越长，越来越贵、越来越慢，最后还会超出模型的上下文窗口。

解决这个问题，可以借鉴我们自己的学习过程：每学完一个章节，我们会复习一下，总结出重要内容。这样，以后需要的时候，我们只需要回头翻阅这些总结就可以了。

对于大模型的记忆，我们可以：

- **较早的对话**，交给大模型写成一段简短的摘要；
- **最近的几条消息**，原样保留。

这样，记忆的长度就不会无限增长了。而写摘要这件事，正好可以交给大模型自己来做。

In [1]:
import os
from abc import ABC, abstractmethod
from enum import Enum
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

## 消息类

### 角色

In [2]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

### 抽象类

In [3]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

### 系统消息类

In [4]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

### 用户消息类

In [5]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

### 助手消息类

In [6]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

In [7]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

## 记忆类

我们在记忆类里添加记忆压缩的能力。

首先我们增加了 4 个记忆类的初始化参数用于设置压缩的策略：

- `provider`：压缩要用到大模型，所以记忆类需要知道用哪个模型；
- `max_messages`：消息超过多少条就压缩；
- `max_tokens`：内容超过多长就压缩；
- `keep_messages`：压缩时，最近的几条消息原样保留，不写进摘要。

为了在演示中马上看到效果，这里的数值设得很小：超过 3 条消息就压缩，只保留最近 1 条。真实的应用里，通常会设成几十条消息、几千个 token。

其他几个函数也需要相应调整：

- `add`：每加一条消息，就检查一下是否超过了限制，超过了就调用 `summarize`；
- `summarize`：实现记忆压缩的核心：
  1. 把消息分成两部分：较早的 `to_summarize`，和最近的 `to_keep`；
  2. 把较早的消息拼成一段文字，加上“请把下面的对话压缩成一段简短的摘要”，交给大模型。这里用了一条专门的系统消息 `SUMMARIZE_SYSTEM_MSG`，让大模型扮演“摘要助手”；
  3. 用摘要替换掉较早的消息：新的记忆 = 一条摘要 + 最近的消息；
- `token_count`：新函数，用于估算记忆有多长。准确计算 token 需要用到模型的分词器，这里简单地数字符个数，作为粗略的估计；
- `note`：新函数，用于演示压缩结果。它把信息缩进两格，前面加一个 `*`，多行的内容也会对齐。

In [8]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1):
        self.provider = provider
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, message: Message):
        self.messages.append(message)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        self.note("SUMMARY", self.history())

    def token_count(self):
        return sum(len(msg.summary() or "") for msg in self.messages)

    @staticmethod
    def note(label: str, text: str):
        lines = str(text).splitlines() or [""]
        print(f"  * {label}:")
        for line in lines:
            print(f"  * {line}")

## 大模型调用器

### 基础类

In [9]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

### OpenAI 调用器

In [10]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

### Claude 调用器

In [11]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

## 智能体类

智能体类也需要改动一个地方：创建记忆时，要把 `provider` 传给 `Memory`，因为记忆压缩要用到大模型。

In [12]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage):
        self.provider = provider
        self.system = system
        self.memory = Memory(provider)

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            user_message = UserMessage(message)
            response = self.provider.call(system or self.system, self.memory.messages + [user_message])
            self.memory.add(user_message)
            self.memory.add(response)
            print(f"AI: {response.summary()}")

## 大模型调用

这次我们准备了三个问题，来看一看会不会触发记忆压缩，以及压缩后的模型是不是还能保持记忆：先自我介绍，再问一个别的问题，最后看看智能体还记不记得小明的名字。

注意看运行结果里缩进的 `* summary` 那几行：那就是记忆被压缩后的效果。

In [13]:
provider = OpenAIProvider()
agent = Agent(provider, system_message)
agent.chat(["你好，我叫小明，正在学这本书。", "第 1 部分讲了什么？", "我叫什么名字？"])


YOU: 你好，我叫小明，正在学这本书。
AI: 你好，小明，很高兴一起学习《从大模型到智能体》！

我可以帮你做这些事：
- 讲解书里的概念，尽量讲得通俗易懂
- 帮你梳理每一章的重点
- 用代码示例解释 Agent、工具调用、记忆、规划等内容
- 陪你做练习题，或者一起设计一个小型智能体项目
- 你把不懂的段落发给我，我可以逐句解释

如果你愿意，我们可以从这几种方式开始：
1. 你告诉我你正在学第几章  
2. 你发来一个你看不懂的概念  
3. 我直接给你做一个“从大模型到智能体”的入门路线图

你想先学哪一个？

YOU: 第 1 部分讲了什么？
  * SUMMARY:
  * [User]: Historical Summary: 用户自我介绍并表示正在学习《从大模型到智能体》。助手说明可以帮助讲解书中概念、梳理章节重点、用代码示例解释 Agent 等内容，并询问用户想从哪部分开始。随后用户询问“第 1 部分讲了什么？”。
  * [Assistant]: 第 1 部分通常是在帮你建立“从大模型到智能体”的整体认知，也就是先回答一个核心问题：
  * 
  * **为什么只有大模型还不够，为什么还需要智能体？**
  * 
  * 一般会讲这些内容：
  * 
  * 1. **大模型是什么**
  *    - 大模型能生成文本、回答问题、总结内容
  *    - 但它本质上还是“输入什么，输出什么”的模式
  * 
  * 2. **智能体是什么**
  *    - 智能体不只是回答问题
  *    - 它还能**思考、规划、调用工具、执行任务、根据结果调整下一步**
  * 
  * 3. **大模型到智能体的升级路径**
  *    - 仅靠对话：能聊天
  *    - 加上工具：能查资料、算数据、写代码
  *    - 加上记忆：能记住用户偏好和历史
  *    - 加上规划和反思：能更像一个“会做事”的助手
  * 
  * 4. **为什么要学智能体**
  *    - 因为很多真实任务不是一句话就能完成的
  *    - 比如：订票、写报告、查资料、整理信息、自动化工作流
  *    - 这些都需要“分步骤完成”的能力
  * 
  * 如果你愿意，我可以继续给你：
  * - **第 1 部分的详细分章梳理**


看看压缩以后，记忆里还剩下什么：

In [14]:
print(agent.memory.history())

[User]: Historical Summary: 用户在这段对话中先自我介绍并提到正在学习《从大模型到智能体》，随后询问了第 1 部分的内容。助手概述了这一部分主要在解释“大模型为什么还不够、为什么需要智能体”，并介绍了大模型、智能体、工具调用、记忆、规划等核心概念。最后，用户问了“我叫什么名字？”，但对话中没有提供用户姓名。
[Assistant]: 我还不知道你的名字。  
如果你愿意，可以告诉我，我之后就用你的名字称呼你。


最早的几条消息已经不见了，取而代之的是一段摘要。只要摘要里提到了小明的名字，学习助手就仍然能答出来。显然，这次压缩记忆以后，智能体丢失了这部分记忆。

## 记忆压缩的代价

压缩记忆解决了“越聊越长”的问题，但也有代价：

- **会丢失细节**：摘要是浓缩过的，没写进摘要的内容就真的忘了。比如小明一开始说过“我喜欢简短的解释”，如果摘要里没有这一句，学习助手之后就不会记得；
- **要多花一次调用**：写摘要也要调用大模型，也要花钱。

除了写摘要，还有一种更简单的记忆压缩方法：**只保留最近的 N 条消息**，更早的直接丢掉，叫作“滑动窗口”。它不用额外调用大模型，但丢得更彻底。

## 动手试试

- 把 `max_messages` 改成 6、`keep_messages` 改成 2，多聊几轮，看看压缩发生的时机有什么变化。
- 修改 `SUMMARIZE_USER_MSG`，要求摘要里“必须保留用户的名字、学习进度和偏好”，看看摘要有什么不同。
- 试着实现滑动窗口：在 `summarize` 里不调用大模型，直接 `self.messages = to_keep`。比较一下两种做法的效果。

## 小结

- 记忆太长时，可以像复习一样：把较早的对话交给大模型写成**摘要**，最近的几条消息则原样保留；
- `Memory` 通过 `max_messages` 和 `max_tokens` 判断什么时候压缩，用 `keep_messages` 决定保留多少条原始消息；
- 摘要以一条带前缀的 `UserMessage` 放在记忆最前面，既保证了格式，也让大模型知道这是摘要；
- 压缩会丢失细节，也会多花一次调用。另一种简单的做法是滑动窗口。

现在学习助手能聊得很久了，但是记忆在内存里。退出程序，所有记忆就都没了。下一章，我们把记忆**保存到文件里**，下次打开时就可以接着聊。